# Estatística I - Aluno

Open In Colab

Notebook de análise exploratória e preparação dos dados para churn.

In [ ]:
# Mount Google Drive
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# Define the path to the CSV file
file_path = '/content/drive/MyDrive/Business_Analytics_Fraude/Estatistica/clientes_base.csv'

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
print('First 5 rows of the DataFrame:')
display(df.head())

# Display the DataFrame's information, including column types
print('\nDataFrame Info (column types and non-null counts):')
df.info()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

categorical_cols = ['plano', 'ciclo', 'regiao', 'canal_aquisicao']

for col in categorical_cols:
    print(f"\n--- Análise da coluna: '{col}' ---")
    value_counts = df[col].value_counts()
    print(value_counts)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=value_counts.index, y=value_counts.values, palette='viridis')
    plt.title(f'Distribuição de {col}')
    plt.xlabel(col)
    plt.ylabel('Contagem')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Quantidade de valores nulos por coluna
print('Quantidade de valores nulos por coluna:')
display(df.isnull().sum())

In [ ]:
# 2. Exemplo de DataFrame quando a coluna 'regiao' é '?'
print("Exemplos onde a coluna 'regiao' é '?':")
display(df[df['regiao'] == '?'].head())

In [ ]:
# 3. Padronizar valores em 'plano'
df['plano'] = df['plano'].str.lower()
df['plano'] = df['plano'].replace({
    'basico': 'básico',
    'padrao': 'padrão',
    'premium': 'premium'
})
print("Valores únicos da coluna 'plano' após a padronização:")
display(df['plano'].value_counts())

In [ ]:
# 4. Padronizar valores em 'canal_aquisicao'
df['canal_aquisicao'] = df['canal_aquisicao'].str.lower()
df['canal_aquisicao'] = df['canal_aquisicao'].replace({
    'organico': 'orgânico',
    'indicacao': 'indicação',
    'ig': 'instagram',
    'instagram': 'instagram',
    'google': 'google'
})
print("Valores únicos da coluna 'canal_aquisicao' após a padronização:")
display(df['canal_aquisicao'].value_counts())

In [ ]:
import numpy as np

# Substituir '?' por NaN em todo o DataFrame
df = df.replace('?', np.nan)
print("Quantidade de valores nulos por coluna após a substituição de '?':")
display(df.isnull().sum())

In [ ]:
date_columns = ['data_assinatura', 'data_cancelamento', 'data_ultimo_acesso', 'data_extracao']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"Coluna '{col}' convertida para datetime.")

print('\nDataFrame Info (tipos de coluna atualizados):')
df.info()

In [ ]:
from itertools import combinations

datetime_cols = df.select_dtypes(include=['datetime64[ns]']).columns.tolist()
print("Colunas datetime identificadas:", datetime_cols)

for col1, col2 in combinations(datetime_cols, 2):
    new_col_name = f'diff_{col1}__{col2}'
    df[new_col_name] = (df[col2] - df[col1]).dt.days
    print(f"Coluna '{new_col_name}' criada.")

print('\nDataFrame com as novas colunas de diferença de datas:')
display(df.head())

In [ ]:
# Remoção de valores inconsistentes
df = df[df['diff_data_assinatura__data_ultimo_acesso'].isnull() | (df['diff_data_assinatura__data_ultimo_acesso'] >= 0)].copy()
df_filtrado = df[(df['idade'] >= 18) & (df['idade'] <= 90)].copy()

print(f"Linhas após limpeza: {len(df_filtrado)}")
display(df_filtrado.head())

In [ ]:
# Feature engineering
df_filtrado['recencia'] = df_filtrado['diff_data_ultimo_acesso__data_extracao']
df_filtrado['meses_assinatura'] = np.where(
    df_filtrado['data_cancelamento'].isnull(),
    (df_filtrado['data_extracao'] - df_filtrado['data_assinatura']).dt.days / 30,
    (df_filtrado['data_cancelamento'] - df_filtrado['data_assinatura']).dt.days / 30
)
df_filtrado['receita_acumulada'] = df_filtrado['valor_mensal'] * df_filtrado['meses_assinatura']

display(df_filtrado[['recencia', 'meses_assinatura', 'receita_acumulada']].head())

In [ ]:
# Preparar subset para análise de churn
df_analise = df_filtrado[(df_filtrado['diff_data_assinatura__data_extracao'] >= 220)].copy()

numerical_cols = ['idade', 'valor_mensal', 'nps', 'dias_ativos_ult_mes', 'horas_assistidas_total', 'recencia', 'meses_assinatura', 'receita_acumulada']
categorical_cols = ['plano', 'ciclo', 'regiao', 'canal_aquisicao']

print("Variáveis numéricas para análise de churn:")
print(numerical_cols)
print("\nVariáveis categóricas para análise de churn:")
print(categorical_cols)

In [ ]:
for col in categorical_cols:
    print(f"\n{'='*60}\nCruzamento: '{col}' vs. 'churn'\n{'='*60}")
    taxa = pd.crosstab(df_analise[col], df_analise['churn'], normalize='index') * 100
    display(taxa.round(1))

In [ ]:
for col in numerical_cols:
    print(f"--- {col} ---")
    display(df_analise.groupby('churn')[col].describe())
    print()

In [ ]:
# Histograma de densidade por churn
df_analise['churn_label'] = df_analise['churn'].map({0: 'Não Churn', 1: 'Churn'})

for col in numerical_cols:
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df_analise, x=col, hue='churn_label', kde=True, stat='density', common_norm=False, palette='viridis')
    plt.title(f'Distribuição de {col} por Churn')
    plt.xlabel(col)
    plt.ylabel('Densidade')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

df_analise.drop(columns=['churn_label'], inplace=True, errors='ignore')